# Parte 1 – Scraping de decretos de emergencia

In [1]:
import time, re
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

#### Paso 1 – robots.txt de gob.pe

In [2]:
headers = {"User-Agent": "Mozilla/5.0"}

robots = requests.get("https://www.gob.pe/robots.txt", headers=headers, timeout=30)
print(robots.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



##### Análisis del robots.txt de gob.pe
**¿Qué prohíbe?** Para cualquier bot (`User-agent: *`), prohíbe acceder a la sección `/admin/` y `sheet=`. Además, bloquea por completo el sitio para `AhrefsBot`.

**¿Está permitida la página de búsquedas?** La ruta `/busquedas` no aparece en ninguna regla `Disallow`, por lo que se conluye que esta permitida.

**¿Por qué igual conviene hacer pausas?** Aunque no se nos exige un `Crawl-delay`, conviene hacer pausas para: 1. evitar que nos bloqueen y 2. Saturar el servicio que brinda gob.pe, porque ademas afectaria a los ciudadanos. Por otro lado, la pagina sí le pide a otros bots `Crawl-delay`: 5 segundos a `GPTBot` y 2 segundos a `OAI-SearchBot`.

#### Paso 2 – Abrir con Selenium la búsqueda de un mes

In [25]:
BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
        "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

url_mes = BASE + "&desde=01-01-2024&hasta=31-01-2024"

navegador = webdriver.Chrome()
navegador.get(url_mes)

WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article")))
time.sleep(3)  # margen extra: la página sigue dibujando resultados
print("Contenido cargado")
print("Artículos encontrados:", len(navegador.find_elements(By.TAG_NAME, "article")))

Contenido cargado
Artículos encontrados: 11


#### Paso 3 – Leer el total de resultados ("N Resultados")

In [26]:
# el total aparece arriba de la lista como "N Resultados", dentro de <h2 role="status"><span>
contador = WebDriverWait(navegador, 30).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "h2[role='status'] span")))
texto_total = contador.text                             # ej. "11 Resultados"
total_resultados = int(re.sub(r"\D", "", texto_total))  # nos quedamos solo con los dígitos

print("Texto en la página:", texto_total)
print("Total de resultados:", total_resultados)

Texto en la página: 11 Resultados
Total de resultados: 11


#### Paso 4 – Extraer los datos de cada resultado

In [27]:
def extraer_articulo(art):
    # enlace principal: trae el número de la norma y su URL
    link = art.find_element(By.CSS_SELECTOR, "a.link-principal")

    # la fecha es la línea que empieza con "Publicado:"
    fecha = ""
    for linea in art.text.split("\n"):
        if linea.startswith("Publicado:"):
            fecha = linea.replace("Publicado:", "").strip()

    # el título es el párrafo debajo de la fecha (viene cortado con "...")
    titulo = art.find_element(By.TAG_NAME, "p").text

    return {
        "numero": link.text.strip(),
        "fecha": fecha,
        "titulo": titulo.strip(),
        "enlace": link.get_attribute("href"),
    }

In [28]:
# prueba con los resultados de enero que ya están abiertos en el navegador
articulos = navegador.find_elements(By.TAG_NAME, "article")
filas_enero = []
for art in articulos:
    filas_enero.append(extraer_articulo(art))

print("Normas extraídas:", len(filas_enero))
pd.DataFrame(filas_enero)

Normas extraídas: 11


,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
